In [93]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver



In [94]:
class PrepState(TypedDict):
    action: str
    status: str



In [95]:
def human_approval(state: PrepState):
    decision = interrupt({
        "message" : "required your approval",
        "action": state["action"]
    })

    if decision == "approve":
        return {"status" : "approve"}
    return {"status" : "reject"}


In [96]:
def approve (state: PrepState):
    print("executing")
    return {"status": "approve"}

def reject (state: PrepState):
    print("rejecting")
    return {"status": "reject"}

def route(state: PrepState):
    if state["status"] == "approve":
        return "approve"
    return "reject"

In [97]:
graph = StateGraph(PrepState)

graph.add_node("human_approval", human_approval)
graph.add_node("approve", approve)
graph.add_node("reject", reject)

graph.add_edge(START, "human_approval")
graph.add_conditional_edges("human_approval", route, {"approve" : "approve", "reject" : "reject"})

graph.add_edge("approve", END)
graph.add_edge("reject", END)


checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [99]:
config = {
    "configurable": {
        "thread_id": "prep-001"
    }
}

res = workflow.invoke({"action" : "Reset Mumbai restaurant shift"}, config=config)

decision = input("Please approve/reject:").lower().strip()



result = workflow.invoke(
    Command(resume=decision),
    config=config
)

result

rejecting


{'action': 'Reset Mumbai restaurant shift', 'status': 'reject'}